# Notebook 03 — General Concepts in Machine Learning
In the previous notebook we studied gradient descent in depth. Let's warm up with a simple example to refresh our memory.

Define

$$h(x) = (x-3)^2 + 1, \qquad x^{(0)} = 10.$$


Remember that gradient descent performs **optimization** over the parameters we can control. Here our goal is to minimize $h(x)$ by adjusting $x$. In the previous notebook, we minimized the loss function by adjusting the weights.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

h = lambda x: (x - 3) ** 2 + 1
x0 = 10

x = np.linspace(-2, 12, 400)

plt.figure(figsize=(7, 5))
plt.plot(x, h(x), color="tab:purple", label=r"$h(x) = (x-3)^2 + 1$")
plt.scatter([x0], [h(x0)], color="black", zorder=5, label=fr"initial x = {x0}")
plt.scatter([3], [h(3)], color="green", zorder=5, label="minimum (3, 1)")

plt.xlabel("x")
plt.ylabel("h(x)")
plt.title("h(x) = $(x-3)^2 + 1$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


Let's run gradient descent by hand. Remember that the update rule is

$$x^{(t+1)} \leftarrow x^{(t)} - \eta\, \frac{dh^{(t)}}{dx}.$$

We start by computing the derivative of $h$ with respect to $x$,

$$\frac{dh}{dx} = 2(x-3).$$

We're given $x^{(0)} = 10$, so the derivative at that point is

$$\frac{dh^{(0)}}{dx} = 2(10-3) = 14.$$

Let's pick a learning rate $\eta = 0.1$ and compute the first update,

$$x^{(1)} = x^{(0)} - \eta\, \frac{dh^{(0)}}{dx} = 10 - 0.1 \cdot 14 = 8.6.$$

Evaluating the derivative again at the new point gives

$$\frac{dh^{(1)}}{dx} = 2(8.6-3) = 11.2,$$

which leads to the second update,

$$x^{(2)} = x^{(1)} - \eta\, \frac{dh^{(1)}}{dx} = 8.6 - 0.1 \cdot 11.2 = 7.48.$$

Computing a few more,

$$\begin{aligned}
\frac{dh^{(2)}}{dx} &= 2(7.48-3) = 8.96,   & x^{(3)} &= x^{(2)} - \eta\, \frac{dh^{(2)}}{dx} = 7.48 - 0.1 \cdot 8.96 = 6.584, \\
\frac{dh^{(3)}}{dx} &= 2(6.584-3) = 7.168, & x^{(4)} &= x^{(3)} - \eta\, \frac{dh^{(3)}}{dx} = 6.584 - 0.1 \cdot 7.168 = 5.8672, \\
\frac{dh^{(4)}}{dx} &= 2(5.8672-3) = 5.7344, & x^{(5)} &= x^{(4)} - \eta\, \frac{dh^{(4)}}{dx} = 5.8672 - 0.1 \cdot 5.7344 = 5.29376, \\
\vdots\ \ & & \vdots\ \ & \\
\frac{dh^{(49)}}{dx} &= 2(3.000125-3) = 0.00025, & x^{(50)} &= 3.000125 - 0.1 \cdot 0.00025 = 3.0001.
\end{aligned}$$

So we repeat for a total of 50 steps in this example. See the below plot for the trajectory of $x$ as it rolls down the curve of $h(x)$.

In [ ]:
h = lambda x: (x - 3) ** 2 + 1
dh = lambda x: 2 * (x - 3)

eta = 0.1
x = 10.0
n_steps = 50

xs = [x]
for t in range(n_steps):
    x = x - eta * dh(x)
    xs.append(x)

for t in [0, 1, 2, 3, 4, 5]:
    print(f"x^({t}) = {xs[t]:.6f},  dh/dx = {dh(xs[t]):+.6f}")
print("   ...")
for t in [n_steps - 1, n_steps]:
    print(f"x^({t}) = {xs[t]:.6f},  dh/dx = {dh(xs[t]):+.6f}")
print(f"\nConverged to x = {xs[-1]:.6f} (true minimum at x = 3)")

xs = np.array(xs)
hs = h(xs)

plt.figure(figsize=(7, 5))
grid = np.linspace(-2, 12, 400)
plt.plot(grid, h(grid), color="tab:purple", label=r"$h(x) = (x-3)^2 + 1$")
plt.plot(xs, hs, "o-", color="red", ms=4, lw=1, alpha=0.6, label="GD steps")
plt.scatter([xs[0]], [hs[0]], color="black", s=120, ec="black", zorder=6,
            label=fr"start $x^{{(0)}} = {xs[0]:.0f}$")
plt.scatter([3], [h(3)], color="green", zorder=5, label="minimum (3, 1)")
plt.xlabel("x"); plt.ylabel("h(x)")
plt.title("Gradient descent (GD) walking down the parabola")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()


Notice that the true minimizer of $h(x) = (x-3)^2 + 1$ is $x^\star = 3$, where the minimum value is $h(3) = 1$. After 50 steps we land very close to it, with $x^{(50)} = 3.0001$ and $h(x^{(50)}) = 1.00000001$.

## Generalization
Gradient descent is a powerful tool for training a model so that it minimizes the loss. Let us recall how we defined the loss, in particular the mean squared error (MSE):

$$
\text{MSE}(\mathbf{y}, \hat{\mathbf{y}})
= \underbrace{\frac{1}{N} \sum_{i=1}^N}_{\text{mean}}
\underbrace{\left(\underbrace{y_i - \hat{y}_i}_{\text{error}}\right)^2}_{\text{squared error}}.
$$

Earlier we argued that empirical risk minimization is the core principle of machine learning. The idea was that a model which does well on the training data (the past) is more likely to do well on future data. This is true in general, but it comes with an important caveat. We should be suspicious of a model that does extremely well on the training data, for reasons we'll now explore.

Our previous toy dataset had only 10 data points (10 houses). Now suppose instead that we have a list of the prices of every house in the entire world. Let's visualize this.

In [ ]:
rng = np.random.default_rng(0)

N = 995
MAX_SIZE = 4.0

size = rng.gamma(shape=2.0, scale=1.0, size=N)
size = size[size <= MAX_SIZE]
while len(size) < N:
    extra = rng.gamma(shape=2.0, scale=1.0, size=N)
    size = np.append(size, extra[extra <= MAX_SIZE])
size = size[:N]

price = 150 + 120 * size + rng.normal(0, 40, size=N)

size_out = rng.uniform(size.min(), MAX_SIZE, size=5)
price_out = (150 + 120 * size_out) + rng.normal(0, 350, size=5)
size = np.append(size, size_out)
price = np.append(price, price_out)

plt.figure(figsize=(7, 5))
plt.scatter(size, price, s=10, alpha=0.4, color="gray")
plt.xlabel("Size (1000s sq ft)")
plt.ylabel("Price ($1000s)")
plt.title(f"Dataset of prices of all houses in the entire world")
plt.grid(True, alpha=0.3)
plt.show()


Since we (hypothetically) have a list of the prices of every house in the world, this scenario contains both the past and the future data. No one can ask us something new about the future (a house we have not seen before), because our dataset already contains all possible houses. Now, let us pretend that the future data is hidden from us and that we have access only to a smaller dataset on which to train our model. This subset will play the role of the past data (the training set) and is highlighted in red below.

In [ ]:
n_total = len(size)
outlier_idx = np.arange(N, n_total)

n_train = 100
train_idx = rng.choice(N, size=n_train - len(outlier_idx), replace=False)
train_idx = np.concatenate([train_idx, outlier_idx])

is_train = np.zeros(n_total, dtype=bool)
is_train[train_idx] = True

size_train, price_train = size[is_train], price[is_train]
print(f"Training set: {is_train.sum()} houses")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

ax1.scatter(size[~is_train], price[~is_train], s=10, alpha=0.3, color="tab:orange", label="Other houses")
ax1.scatter(size_train, price_train, s=10, alpha=0.4, color="tab:blue", label="Training set")
ax1.set_title("House prices with training set highlighted")
ax1.set_xlabel("Size (1000s sq ft)"); ax1.set_ylabel("Price ($1000s)")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.scatter(size_train, price_train, s=10, alpha=0.4, color="tab:blue", label="Training set")
ax2.set_title("Training set only")
ax2.set_xlabel("Size (1000s sq ft)")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Let's train a model! We do not have to use a linear model. In fact, linear models are quite limited in capacity. Let's instead train a much more complex model and fit our training dataset *perfectly*.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(size_train.reshape(-1, 1), price_train)

train_mse = np.mean((model.predict(size_train.reshape(-1, 1)) - price_train) ** 2)
print(f"MSE on the training set: {train_mse:.4f}  (fits the training data perfectly)")

grid = np.linspace(size_train.min(), size_train.max(), 2000).reshape(-1, 1)
pred = model.predict(grid)

plt.figure(figsize=(8, 5))
plt.scatter(size_train, price_train, s=20, color="tab:blue", alpha=0.5, zorder=3, label="Training data")
plt.plot(grid, pred, color="black", alpha=0.6, lw=1.2, label="Trained model")
plt.xlabel("Size (1000s sq ft)")
plt.ylabel("Price ($1000s)")
plt.title("An over-complex model fitting the training data almost perfectly")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

We have exactly 0 MSE on the training set. That seems great, right? Not really. We should never be satisfied by performance on the training set alone. Remember that our entire purpose is to do well on the future data. Let's check that now.

In [ ]:
size_rest, price_rest = size[~is_train], price[~is_train]

rest_mse = np.mean((model.predict(size_rest.reshape(-1, 1)) - price_rest) ** 2)
print(f"MSE on the training set:    {train_mse:.4f}")
print(f"MSE on the future data:    {rest_mse:.2f}")

plt.figure(figsize=(8, 5))
plt.scatter(size_rest, price_rest, s=10, alpha=0.4, color="tab:orange", label="held-out data (not trained on)")
plt.plot(grid, pred, color="black", alpha=0.6, lw=1.2, label="trained model")
plt.xlabel("size (1000s sq ft)")
plt.ylabel("price ($1000s)")
plt.title("The over-complex model vs. the data it never saw")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

Our model fails badly on the data it has never seen before. The MSE on the future data is huge. This phenomenon is called **overfitting**. Instead of learning the underlying relationship between size and price, the model has simply **memorized** the training data and now behaves like a lookup table.

Overfitting is very common in machine learning. The more complex the model, the easier it is to overfit. Let us see how the performance on future data compares if we train a plain linear model instead.

In [ ]:
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(size_train.reshape(-1, 1), price_train)

grid_x = np.linspace(size_train.min(), size_train.max(), 200)
line = linear.predict(grid_x.reshape(-1, 1))

lin_train_mse = np.mean((linear.predict(size_train.reshape(-1, 1)) - price_train) ** 2)
lin_test_mse = np.mean((linear.predict(size_rest.reshape(-1, 1)) - price_rest) ** 2)
print(f"Linear model  MSE on the training set: {lin_train_mse:.2f}")
print(f"Linear model  MSE on the future data:  {lin_test_mse:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

ax1.scatter(size_train, price_train, s=20, color="tab:blue", alpha=0.6, zorder=3, label="Training data")
ax1.plot(grid_x, line, color="black", lw=2, label="Linear model")
ax1.set_title("Linear model vs. training set")
ax1.set_xlabel("Size (1000s sq ft)"); ax1.set_ylabel("Price ($1000s)")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.scatter(size_rest, price_rest, s=10, alpha=0.4, color="tab:orange", label="Future data")
ax2.plot(grid_x, line, color="black", lw=2, label="Linear model")
ax2.set_title("Linear model vs. future data")
ax2.set_xlabel("Size (1000s sq ft)")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Inspect the numbers closely (you may get slightly different numbers).
- The complex model had 0 MSE on the training set but 5543.20 MSE on the future data.
- The linear model has 3207.79 MSE on the training set and 1555.79 MSE on the future data.

That is roughly a 3 times improvement on the MSE on future data, even though the simpler model fits the training data worse!

Here are a few more analogies to help build intuition for overfitting.

1. Imagine you are a student preparing for an exam. You have a set of practice questions (your training data) and you want to do well on the actual exam (future data). If you memorize the answers to the practice questions without understanding the underlying concepts, you might ace those specific questions yet fail on new ones that test real understanding.

2. Imagine you are learning to drive but you only ever practice one single route from home to school. You might become an expert on that route, knowing exactly where to turn, which lights are usually red, and where the potholes are. This is memorization rather than understanding. The day you are asked to drive somewhere new you struggle, because you never learned the general skill of driving, such as reading signs, reacting to traffic, and judging distances.

3. Imagine you have trained a model to classify images of cats and dogs, so that given an image it tells you which one it is. Suppose every cat in the training set happens to be orange. An overfitted model will learn to associate "orange" with "cat". When you later show it a mandarin, it thinks it is a cat, because it never learned the underlying idea of what makes a cat a cat. It only memorized the training data.

### Underfitting
Overfitting is the failure mode of a model that is too complex. The opposite failure mode is **underfitting**, where a model is too simple to capture the real pattern in the data.

To see this clearly, let's build a dataset whose true relationship is a cosine wave and then try to fit it with a straight line. A line simply does not have the flexibility to follow an up-and-down curve, so it misses the pattern on both the training data and the future data.

For comparison we also fit a more complex nonlinear model, which has enough capacity to bend along with the cosine. The linear model underfits, while the complex one captures the true shape.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)

x = rng.uniform(0, 2 * np.pi, size=200)
y = np.cos(x) + rng.normal(0, 0.1, size=x.shape)

is_train = rng.random(size=x.shape) < 0.5
x_train, y_train = x[is_train], y[is_train]
x_rest,  y_rest  = x[~is_train], y[~is_train]

linear = LinearRegression().fit(x_train.reshape(-1, 1), y_train)

poly = make_pipeline(PolynomialFeatures(degree=6), LinearRegression())
poly.fit(x_train.reshape(-1, 1), y_train)

def mse(model):
    train = np.mean((model.predict(x_train.reshape(-1, 1)) - y_train) ** 2)
    rest  = np.mean((model.predict(x_rest.reshape(-1, 1))  - y_rest)  ** 2)
    return train, rest

lin_train, lin_rest = mse(linear)
poly_train, poly_rest = mse(poly)
print(f"Linear model   MSE  train: {lin_train:.4f}   Future: {lin_rest:.4f}")
print(f"Complex model  MSE  train: {poly_train:.4f}   Future: {poly_rest:.4f}")

grid = np.linspace(0, 2 * np.pi, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, model, name in [(ax1, linear, "Linear model (underfits)"),
                        (ax2, poly, "Complex model (good fit)")]:
    ax.scatter(x_rest, y_rest, s=10, alpha=0.5, color="tab:orange", label="Future data")
    ax.scatter(x_train, y_train, s=15, alpha=0.5, color="tab:blue", label="Training data")
    ax.plot(grid, model.predict(grid.reshape(-1, 1)), color="black", lw=2, label=name)
    ax.set_xlabel("x"); ax.set_title(name)
    ax.legend(); ax.grid(True, alpha=0.3)
ax1.set_ylabel("y")
plt.tight_layout()
plt.show()


### Key takeaway
Model complexity is a balancing act. A model that is too simple underfits, missing the real pattern and doing poorly on both the past and the future. A model that is too complex overfits, memorizing the training data and doing poorly on the future. The goal is the middle ground, a model with just enough capacity to capture the true relationship without latching onto noise.

This is why we never judge a model by its training performance alone. The number that actually matters is the error on data the model has never seen, since that is what tells us how well it will generalize to the future.

## Few more words

### Sample and population
The **population** is the full set of cases we ultimately care about, such as every house in the world. We almost never get to see all of it. Instead we work with a **sample**, which is a smaller subset of the population that we actually have data for. Our training data is a sample, and all of machine learning is really an attempt to learn something about the population from the limited sample in front of us. Overfitting is what happens when a model describes the quirks of its particular sample rather than the pattern shared by the whole population.

Sometimes, the task of learning from a sample becomes even harder when the sample is **biased**. Consider the following example. Suppose the population consists of all 8 billion people in the world, and we want to select 1,000 of them to create a training dataset. If we choose those 1,000 people **uniformly at random** from the population (so that each person has a $\frac{1}{8,000,000,000}$ chance of being selected), then we have a good chance of obtaining a representative sample. However, if we select those 1,000 people by going to a single city and surveying the first 1,000 people we encounter, the sample is likely to be biased.

> **Note:** Sampling bias is not always so obvious! My favorite example is the famous 1936 Literary Digest poll. Give it a read here: https://mathcenter.oxford.emory.edu/site/math117/historicalBlunders/

### Train, validation, and test splits
So far we have talked about past data and future data as if there were only two groups. In practice we usually split our available sample into three parts.

1. The **training set** is the data the model learns from. The model adjusts its parameters (its weights) to fit these points.
2. The **validation set** is held out from training and used to **tune** hyperparameters, such as the learning rate or how complex the model should be. We pick a learning rate, look at the performance on the validation set, and repeat until we find a good one. The model only indirectly sees the validation data (through our hyperparameter choices).
3. The **test set** is held out from everything and looked at only once at the very end. It gives an honest estimate of how the final model will perform on genuinely new data.

The reason for keeping these datasets separate is the main lesson of this notebook: once the model has access to a set of examples, directly or indirectly, it can begin to fit them. The training set is used to learn the model parameters. The validation set is kept separate so that we can compare different hyperparameter choices on examples that were not used during training. However, there is still a subtle form of overfitting. Every time we choose a hyperparameter because it performs well on the validation set, we are using information from those validation examples to guide our decisions. Over many rounds of experimentation, the model-development process can gradually become tailored to the validation set itself. This is why, for a true future, we need a third dataset, i.e. the test set. By keeping the test set completely untouched until the very end, we ensure that neither the model parameters nor the hyperparameter choices have been influenced by those examples. As a result, test performance provides our best estimate of how the model will perform on truly unseen future data.

As a final word, let's bring these concepts together with a very concrete example.

> The US Forest Service wants to build a machine learning model that detects wildfires from drone images, so that given an image it predicts whether or not the image contains a flame. They hand you a dataset of 1000 images, each labeled as "fire" or "no fire". This dataset is your sample. You split it into 700 training images, 100 validation images, and 200 test images. You then design a model, say a linear model, and train it on the training set. You try a few different learning rates, and you also try a few different model designs such as increasingly complex models or a neural network, and you use the validation set to decide which one works best. Once you have settled on a final model, you evaluate it (calculate errors) on the test set to estimate how well it will do on new drone images in the future. You report the test set performance (not training or validation) to the Forest Service so they can decide whether to deploy it in their wildfire detection system.
>
> The one thing you must never do is the following. You look at the test set performance, then go back and tweak your model until that number looks good. This is cheating, because you would be using the test set to make decisions about the model. The test set is there to imitate the future (which we don't have access to). Once the test set has influenced your choices, it is no longer untouched data, and its number no longer reflects future performance honestly.

## Summary

- The real goal of learning is **generalization**: doing well on new, unseen data, not just the training data. A low training loss is not enough on its own.
- **Overfitting** happens when a model is too complex. Instead of learning the underlying pattern, it memorizes the training data (noise and all) and behaves like a lookup table, so it scores near-perfectly on the training set but poorly on future data.
- **Underfitting** is the opposite failure: a model too simple to capture the real pattern, so it does poorly on both the training data and future data.
- The data we train on is a **sample** drawn from a larger **population** we actually care about. We learn about the population from the limited sample in front of us.
- In practice we split our sample into three parts: a **training set** (the model learns from it), a **validation set** (used to tune hyperparameters), and a **test set** (looked at only once, to honestly estimate future performance). The test set must never influence our choices, or it stops reflecting the future.